# Real-Data End-to-End Notebook Workflow

Bu notebook, framework'ün gerçek bir kullanım akışını gösterir: veri keşfi/ingest -> feature -> strateji/backtest -> signal -> intent -> risk -> execution -> portfolio.

## 1) Environment setup and config/custom parameter selection

Environment setup: notebook'u repo root'tan çalıştırın ve parametreleri burada değiştirin.

In [ ]:
from __future__ import annotations

import json
import ssl
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

from src.algotradeplan.backtest import RealisticBacktester
from src.algotradeplan.core.intent import signal_to_intent
from src.algotradeplan.orchestration.real_data_autopilot import run_real_data_autopilot
from src.algotradeplan.orchestration.trade_flow import TradeFlow
from src.algotradeplan.plugins.connectors.simulated_fill_connector import SimulatedFillExecutionConnectorPlugin
from src.algotradeplan.plugins.data.market import CcxtMarketDataAgent, collect_market_source_data
from src.algotradeplan.plugins.indicators import RollingWindowFeatureEngine
from src.algotradeplan.plugins.risk.engine import RiskEngine
from src.algotradeplan.plugins.strategies import EmaCrossAtrStopStrategyPlugin
from src.algotradeplan.portfolio.manager import PortfolioManager

REPORT_PATH = Path("artifacts/notebooks/real_data_workflow_report.json")
MAX_SYMBOLS = 3
ALLOW_PARTIAL = True
DEFAULT_QTY = 0.01

print({"report_path": str(REPORT_PATH), "max_symbols": MAX_SYMBOLS, "allow_partial": ALLOW_PARTIAL})

## 2) Data/asset discovery and real-source ingestion

Data/asset discovery adımında registry üzerinden kaynakları tarayıp örnek semboller seçiyoruz.

In [ ]:
def _get_json(url: str, params: dict[str, object]):
    query = urlencode({k: v for k, v in params.items() if v is not None})
    request_url = f"{url}?{query}" if query else url
    request = Request(request_url, headers={"Accept": "application/json", "User-Agent": "AlgoTradePlanNotebook/1.0"})
    with urlopen(request, timeout=20, context=ssl.create_default_context()) as response:
        return json.loads(response.read().decode("utf-8"))

def discover_assets(max_symbols: int = MAX_SYMBOLS, allow_partial: bool = ALLOW_PARTIAL):
    results, issues = collect_market_source_data(
        get_json=_get_json,
        max_symbols=max_symbols,
        allow_partial=allow_partial,
    )
    discovered = [{"source": r.source, "selected_asset": r.selected_asset, "asset_count": r.asset_count} for r in results]
    return discovered, issues

def fetch_binance_klines(symbol: str, limit: int = 200):
    agent = CcxtMarketDataAgent()
    datasets = agent.fetch_exchange_datasets("binanceusdm", symbol)
    return datasets.get("kline", [])[-limit:]

assets, source_issues = discover_assets()
assets[:5], source_issues[:3]

## 3) Feature engineering, dataset inspection, and visualization

Feature engineering aşamasında OHLCV verisini EMA/ATR/Bollinger ile zenginleştiriyoruz.

In [ ]:
selected_symbol = assets[0]["selected_asset"] if assets else "BTC/USDT"
klines = fetch_binance_klines(selected_symbol, limit=250)

candles = [
    {"open": float(k[1]), "high": float(k[2]), "low": float(k[3]), "close": float(k[4])}
    for k in klines if len(k) >= 5
]

highs = [c["high"] for c in candles]
lows = [c["low"] for c in candles]
closes = [c["close"] for c in candles]
features = RollingWindowFeatureEngine().compute(highs=highs, lows=lows, closes=closes)

{
    "selected_symbol": selected_symbol,
    "candle_count": len(candles),
    "latest_close": closes[-1] if closes else None,
    "ema_fast": features.ema_fast,
    "ema_slow": features.ema_slow,
    "atr": features.atr,
}

## 4) Strategy instantiate, rolling/OOS backtest, and single-asset signal -> intent -> risk -> portfolio

Bu bölümde signal -> intent -> risk -> portfolio zincirini gerçek framework bileşenleriyle çalıştırıyoruz.

In [ ]:
def optimize_windows(closes: list[float], highs: list[float], lows: list[float]):
    strategy = EmaCrossAtrStopStrategyPlugin()
    best_params, summary = strategy.optimize(closes, highs, lows)
    return strategy, best_params, summary

strategy, best_params, best_summary = optimize_windows(closes, highs, lows)
signal = strategy.generate_signal({"candles": candles})

intent = signal_to_intent(
    signal,
    symbol=selected_symbol,
    quantity=DEFAULT_QTY,
    strategy_id=strategy.plugin_id,
    price=closes[-1] if closes else 0.0,
)

risk_engine = RiskEngine(max_notional=2_000.0, max_position_size=1.0)
portfolio = PortfolioManager(starting_cash=10_000.0)
trade_flow = TradeFlow(
    strategy=strategy,
    risk=risk_engine,
    execution=SimulatedFillExecutionConnectorPlugin(),
)

flow_result = trade_flow.run({
    "symbol": selected_symbol,
    "price": closes[-1] if closes else 0.0,
    "quantity": DEFAULT_QTY,
    "candles": candles,
})
portfolio_snapshot = portfolio.apply_execution(flow_result.execution)

{
    "best_params": best_params,
    "backtest": best_summary.to_dict() if hasattr(best_summary, "to_dict") else best_summary,
    "signal": signal,
    "intent": intent.to_dict() if intent else None,
    "risk_decision": flow_result.risk_decision,
    "execution": flow_result.execution,
    "portfolio": portfolio_snapshot,
}

## 5) Portfolio-wide run and metric summary

Portfolio-wide run: birden fazla asset için özet metrik çıkartıyoruz.

In [ ]:
portfolio_results = []
backtester = RealisticBacktester()

for row in assets[:2]:
    symbol = row["selected_asset"]
    sample_klines = fetch_binance_klines(symbol, limit=120)
    sample_candles = [
        {"open": float(k[1]), "high": float(k[2]), "low": float(k[3]), "close": float(k[4])}
        for k in sample_klines if len(k) >= 5
    ]
    if len(sample_candles) < 20:
        continue

    h = [c["high"] for c in sample_candles]
    l = [c["low"] for c in sample_candles]
    c = [c["close"] for c in sample_candles]

    strat, params, _ = optimize_windows(c, h, l)
    positions = strat._positions_from_params(c, h, l, params)
    bt = backtester.run(c, positions)
    portfolio_results.append({
        "symbol": symbol,
        "net_pnl": bt.net_pnl,
        "max_drawdown": bt.max_drawdown,
        "sharpe": bt.sharpe_ratio,
        "trade_count": bt.trade_count,
    })

portfolio_results

## 6) Results and next steps

Aynı workflow'u otomatik smoke pipeline ile tek adımda çalıştırmak için:

- `python scripts/e2e_real_data_smoke.py --interactive --allow-partial`
- veya notebook içinden `run_real_data_autopilot(...)`

In [ ]:
report = run_real_data_autopilot(
    report_path=REPORT_PATH,
    max_symbols_per_source=MAX_SYMBOLS,
    allow_partial=ALLOW_PARTIAL,
)

{
    "report_path": str(REPORT_PATH),
    "market_sources": len(report.market_sources),
    "source_issues": report.source_issues[:3],
    "intent": report.intent,
    "risk_decision": report.risk_decision,
    "portfolio": report.portfolio,
    "backtest": report.backtest,
}